In [64]:
#SETUP
from datetime import datetime
from delta import DeltaTable
import ConnectionConfig as cc
cc.setupEnvironment()

In [65]:
#SETUP
spark = cc.startLocalCluster("dimCustomerIncrementalLoad")
spark.getActiveSession()

## Incremental load 
* Setting the parameters

In [66]:

run_timestamp = datetime.now()

In [67]:
cc.set_connectionProfile("VeloDB")

#### Read source table


In [68]:
#EXTRACT
df_velo_users =spark.read \
    .format("jdbc") \
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "velo_users") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "userid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 20) \
    .load()

df_subscriptions = spark.read \
    .format("jdbc") \
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "subscriptions") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "subscriptionid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 20) \
    .load()

df_subscription_types = spark.read \
    .format("jdbc") \
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "subscription_types") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "subscriptiontypeid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 20) \
    .load()

In [69]:
#TRANSFORM
df_velo_users.createOrReplaceTempView("operational_velo_users")
df_subscriptions.createOrReplaceTempView("operational_subscriptions")
df_subscription_types.createOrReplaceTempView("operational_subscription_types")

In [70]:
#TRANSFORM
df_dim_customers = spark.sql("""
    SELECT 
        monotonically_increasing_id() as source_customer_SK,
        u.userid as source_customer_id,
        u.name as source_customer,
        CONCAT(u.street, ' ', u.number, ', ', u.zipcode, ' ', u.city, ', ', u.country_code) as source_address,
        st.description as source_subscription_type,
        s.validfrom as source_validform,
        md5(concat(u.name, source_address)) as source_md5
    FROM operational_velo_users u
        JOIN operational_subscriptions s 
                ON u.userid = s.userid
        JOIN operational_subscription_types st 
                ON s.subscriptiontypeid = st.subscriptiontypeid
    WHERE s.validfrom = (
        SELECT MAX(validfrom) 
        FROM operational_subscriptions 
        WHERE userid = u.userid)
""")

df_dim_customers.createOrReplaceTempView("dimCustomer_new")

In [71]:
#TRANSFORM
dt_dimCustomer = DeltaTable.forPath(spark, "./spark-warehouse/dimcustomer")
dt_dimCustomer.toDF().createOrReplaceTempView("dimCustomer_current")

#DEBUG CODE TO SHOW CONTENT OF DIMENSION
spark.sql("""
   select * from dimCustomer_new
""").show()

+------------------+------------------+--------------------+--------------------+------------------------+----------------+--------------------+
|source_customer_SK|source_customer_id|     source_customer|      source_address|source_subscription_type|source_validform|          source_md5|
+------------------+------------------+--------------------+--------------------+------------------------+----------------+--------------------+
|                 0|                 2|   van der Zee Julia|bbbbbbbbbb 43 , 2...|                   MAAND|      2021-10-05|b362c982b3a5407e0...|
|                 1|                 4|      Willems Angela|Graaf Joseph de P...|                     DAG|      2019-04-08|12722af9620035dbe...|
|                 2|                 3|     de Boer Ricardo|Maria Clarastraat...|                    JAAR|      2022-09-17|fc0e2a0b6f191c377...|
|                 3|                 1|         Bouman Lars|Kerkstraatstraat ...|                     DAG|      2020-12-14|0c5a1e8

##### 2 DETECT CHANGES


In [72]:
#TRANSFORM
detectedChanges = spark.sql("""
    select * 
    from dimCustomer_new source
        left outer join dimCustomer_current dwh 
            on dwh.customer_id = source.source_customer_id
            and dwh.current == true
    where dwh.customer_id is null or dwh.md5 <> source.source_md5
""")
detectedChanges.createOrReplaceTempView("detectedChanges")

#DEBUG CODE TO SHOW CONTENT OF DETECTED CHANGES
detectedChanges.show()

+------------------+------------------+---------------+--------------------+------------------------+----------------+--------------------+-----------+-----------+-----------+--------------------+-----------------+-------------------+-------------------+----------+--------------------+-------+
|source_customer_SK|source_customer_id|source_customer|      source_address|source_subscription_type|source_validform|          source_md5|customer_SK|customer_id|       name|             address|subscription_type|          scd_start|            scd_end| validform|                 md5|current|
+------------------+------------------+---------------+--------------------+------------------------+----------------+--------------------+-----------+-----------+-----------+--------------------+-----------------+-------------------+-------------------+----------+--------------------+-------+
|                 3|                 1|    Bouman Lars|Kerkstraatstraat ...|                     DAG|      2020-12-

##### 3 TRANSOFRM TO UPSERTS


In [73]:
#TRANSFORM 
df_upsert = spark.sql(f"""
    select  
        source_customer_SK as customer_SK,
        source_customer_id as customer_id,
        source_customer as name,
        source_address as address,
        source_subscription_type as subscription_type,
        to_timestamp('{run_timestamp}') as scd_start,
        to_timestamp('2600-01-01', 'yyyy-MM-dd') as scd_end,
        true as current,
        source_md5 as md5
    FROM detectedChanges
    union all
        select  
            customer_SK,
            customer_id,
            name,
            address,
            subscription_type,
            scd_start,
            to_timestamp('{run_timestamp}') as scd_end,
            false as current,
            md5
        FROM detectedChanges
        where current is not null
""")
df_upsert.createOrReplaceTempView("upsert")

In [74]:
#DEBUG CODE TO SHOW CONTENT OF UPSERT
spark.sql("""
    select * from upsert
""").show()

+-----------+-----------+-----------+--------------------+-----------------+--------------------+--------------------+-------+--------------------+
|customer_SK|customer_id|       name|             address|subscription_type|           scd_start|             scd_end|current|                 md5|
+-----------+-----------+-----------+--------------------+-----------------+--------------------+--------------------+-------+--------------------+
|          3|          1|Bouman Lars|Kerkstraatstraat ...|              DAG|2024-10-13 15:53:...| 2600-01-01 00:00:00|   true|0c5a1e8b8c38f6720...|
|          3|          1|Bouman Lars|Kerkstraat 156 , ...|              DAG| 1999-01-01 00:00:00|2024-10-13 15:53:...|  false|1e174588b6a5f055a...|
+-----------+-----------+-----------+--------------------+-----------------+--------------------+--------------------+-------+--------------------+



In [84]:
#CREATE MERGE 
spark.sql("""
    MERGE INTO dimCustomer_current as target
    using upsert as source on target.customer_id = source.customer_id 
        and source.current = false 
        and target.current = true
    WHEN MATCHED THEN UPDATE SET scd_end = source.scd_end, current = source.current
    WHEN NOT MATCHED THEN INSERT (customer_SK, customer_id, name, address, subscription_type,  scd_start, scd_end, current, md5) values (source.customer_SK, source.customer_id, source.name, source.address, source.subscription_type, source.scd_start, source.scd_end, source.current, source.md5)
""")

#DEBUG CODE TO SHOW CONTENT OF DIMENSION
dt_dimCustomer.toDF().sort("customer_id", "scd_start").show(100)

+-----------+-----------+--------------------+--------------------+-----------------+--------------------+--------------------+----------+--------------------+-------+
|customer_SK|customer_id|                name|             address|subscription_type|           scd_start|             scd_end| validform|                 md5|current|
+-----------+-----------+--------------------+--------------------+-----------------+--------------------+--------------------+----------+--------------------+-------+
|          3|          1|         Bouman Lars|Kerkstraat 156 , ...|              DAG| 1999-01-01 00:00:00|2024-10-13 15:53:...|2020-12-14|1e174588b6a5f055a...|  false|
|          3|          1|         Bouman Lars|Kerkstraatstraat ...|              DAG|2024-10-13 15:53:...| 2600-01-01 00:00:00|      NULL|0c5a1e8b8c38f6720...|   true|
|          0|          2|   van der Zee Julia|bbbbbbbbbb 43 , 2...|            MAAND| 1999-01-01 00:00:00| 2100-12-12 00:00:00|2021-10-05|b362c982b3a5407e0...| 

In [85]:
spark.stop()